# ASOS A/B Testing Exploration

This notebook walks through the ASOS Digital Experiments Dataset to understand how AB testing behaves across conversion, revenue, and engagement metrics. The goal is to compute lift, confidence, and statistical significance for representative challenger variants and then save the visual artifacts for future storytelling.

In [ ]:
import math
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set(style='whitegrid', palette='muted')

BASE_DIR = Path('..')
DATA_PATH = BASE_DIR / 'data' / 'asos_digital_experiments_dataset.csv'
OUTPUTS_DIR = BASE_DIR / 'outputs'
PLOTS_DIR = OUTPUTS_DIR / 'plots'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

## Dataset overview

We read all observed variants, metrics, and cumulative statistics from the CSV. The first cell above shows the raw columns. The code below summarizes how many experiments and metrics are present to orient the analysis.

In [ ]:
overview = pd.DataFrame({
    'distinct_experiments': [df['experiment_id'].nunique()],
    'distinct_metrics': [df['metric_id'].nunique()],
    'total_rows': [len(df)]
})

duration = df.groupby('experiment_id')['time_since_start'].max().describe()
print('Duration (in hours) summary for experiments:')
print(duration.round(1))

display(overview.T)

## From control to challenger variants

Rather than join across variant IDs, we take advantage of the fact that each row already stores control (`mean_c`, `count_c`, `variance_c`) and treatment (`mean_t`, `count_t`, `variance_t`) absolute statistics for that challenger variant. We just need the latest snapshot per experiment, metric, and variant combination.

In [ ]:
analysis_df = (
    df[df['variant_id'] != 1]
    .sort_values('time_since_start')
    .groupby(['experiment_id', 'metric_id', 'variant_id'], as_index=False)
    .last()
)
summary = analysis_df.rename(columns={
    'count_c': 'count_control',
    'count_t': 'count_treatment',
    'mean_c': 'mean_control',
    'mean_t': 'mean_treatment',
    'variance_c': 'variance_control',
    'variance_t': 'variance_treatment',
})

summary['lift'] = summary['mean_treatment'] - summary['mean_control']
summary['relative_lift'] = summary['lift'] / summary['mean_control'].replace(0, np.nan)
summary['std_error'] = np.sqrt(
    summary['variance_control'] / summary['count_control'] +
    summary['variance_treatment'] / summary['count_treatment']
)
summary['z_score'] = summary['lift'] / summary['std_error']

summary['p_value'] = summary['z_score'].map(lambda z: 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2)))))
summary['significant'] = summary['p_value'] < 0.05
summary = summary.replace([np.inf, -np.inf], np.nan).dropna(subset=['std_error'])

summary.head(8)

## Capture final summary

Save the filtered summary and highlight significant lifts. This table is what the Streamlit app and any downstream storytelling will reuse.

In [ ]:
summary_path = OUTPUTS_DIR / 'experiment_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved a cleaned summary to', summary_path.name)

## Visual storytelling: lift by experiment

This grouped view surfaces which experiments produced the largest absolute lift so you can immediately recognize standout results.

In [ ]:
largest_lifts = (
    summary.assign(abs_lift=lambda df: df['lift'].abs())
    .sort_values('abs_lift', ascending=False)
    .head(10)
    .assign(label=lambda df: df['experiment_id'] + ' (metric ' + df['metric_id'].astype(str) + ')')
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=largest_lifts,
    x='lift',
    y='label',
    hue='significant',
    dodge=False,
    palette=['#4c78a8', '#f58518'],
)
plt.axvline(0, color='gray', linewidth=0.8)
plt.title('Top absolute lift across challenger variants')
plt.xlabel('Mean difference (treatment - control)')
plt.ylabel('Experiment (metric)')
plt.legend(title='Statistically significant')
plt.gca().xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:.3f}'))
plt.tight_layout()
plot_path = PLOTS_DIR / 'metric_lift_bar.png'
plt.savefig(plot_path)
plt.close()
print('Saved lift bar chart to', plot_path.name)

## Evolution over time for a representative experiment

Choose the experiment with the most data points and plot both variants for the metric that has the longest run. This visual demonstrates how the treatment curve diverges from the control as the experiment progresses.

In [ ]:
selected_experiment = df['experiment_id'].value_counts().idxmax()
exp_df = df[df['experiment_id'] == selected_experiment]
metric_of_interest = exp_df['metric_id'].mode().iloc[0]
exp_metric = exp_df[exp_df['metric_id'] == metric_of_interest].sort_values('time_since_start')

plt.figure(figsize=(10, 5))
for variant in sorted(exp_metric['variant_id'].unique()):
    variant_df = exp_metric[exp_metric['variant_id'] == variant]
    plt.plot(
        variant_df['time_since_start'],
        variant_df['mean_t'],
        label=f'Variant {variant} (mean treatment)',
        linestyle='-' if variant == 2 else '--',
    )
    plt.plot(
        variant_df['time_since_start'],
        variant_df['mean_c'],
        label=f'Variant {variant} (mean control)',
        linestyle=':' if variant == 2 else '-.',
    )
plt.title(f'Experiment {selected_experiment} · Metric {metric_of_interest} over time')
plt.xlabel('Time since experiment start (hours)')
plt.ylabel('Reported metric')
plt.legend(loc='best', fontsize='small')
plt.tight_layout()
time_plot_path = PLOTS_DIR / 'time_series_selected_experiment.png'
plt.savefig(time_plot_path)
plt.close()
print('Saved time-series chart to', time_plot_path.name)